## Análisis y feature engineering — Seguro de Vida Individual (CNSF 2021–2024)

---

### Dependencias

| Notebook | Rol |
|---|---|
| `prepare_datasets.ipynb` | ETL canónico — debe ejecutarse antes que este notebook |

### Datasets de entrada

| Archivo | Tabla | Filas aprox. |
|---|---|---|
| `data/prepared/emision.parquet` | Cartera expuesta | 3.5 M |
| `data/prepared/siniestros.parquet` | Experiencia de siniestralidad | 297 K |
| `data/prepared/comisiones.parquet` | Gastos de adquisición | 828 K |

### Objetivo

Construir métricas por segmento demográfico y geográfico como inputs para modelos de pricing:
siniestralidad, frecuencia, severidad y prima por asegurado.

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

OUTPUT = Path("../data/prepared")

emision_f    = pd.read_parquet(OUTPUT / "emision.parquet")
siniestros_f = pd.read_parquet(OUTPUT / "siniestros.parquet")
comisiones_f = pd.read_parquet(OUTPUT / "comisiones.parquet")

frames = {"emision": emision_f, "siniestros": siniestros_f, "comisiones": comisiones_f}

for name, df in frames.items():
    print(f"✓ {name:<12} {df.shape}")

✓ emision      (3508589, 12)
✓ siniestros   (297501, 13)
✓ comisiones   (827903, 16)


In [2]:
print("=== EMISION ===")
display(emision_f.head())

print("=== SINIESTROS ===")
display(siniestros_f.head())

print("=== COMISIONES ===")
display(comisiones_f.head())

=== EMISION ===


,ANIO,EDAD,COBERTURA,PLAN_DE_LA_POLIZA,MODALIDAD_DE_LA_POLIZA,MONEDA,ENTIDAD,SEXO,FORMA_DE_VENTA,NUMERO_DE_ASEGURADOS,PRIMA_EMITIDA,SUMA_ASEGURADA
0,2021,22,Exención de pago de prima,Vitalicio,Tradicional,Extranjera,Chihuahua,Femenino,Agentes Persona Física,1,127.68,0.00
1,2021,47,Fallecimiento,Vitalicio,Tradicional,Extranjera,Coahuila,Masculino,Fuerza de Venta Interna o Casa Matriz,6,146947.26,9432861.69
2,2021,34,Fallecimiento,Dotal Mixto,Tradicional,Indizada,Querétaro,Femenino,Agentes Persona Física,64,1206118.07,36899161.66
3,2021,29,Fallecimiento,Temporal,Tradicional,Nacional,Hidalgo,Femenino,Telemercadeo,102,38841.70,13457299.02
4,2021,54,Sobrevivencia,Dotal Mixto,Tradicional,Extranjera,Mexico,Masculino,Otra Forma de Venta,10,15704.26,1148045.70


=== SINIESTROS ===


,ANIO,EDAD,COBERTURA,ENTIDAD,CAUSA_DEL_SINIESTRO,PLAN_DE_LA_POLIZA,MODALIDAD_DE_LA_POLIZA,SEXO,NUMERO_DE_SINIESTROS,MONTO_RECLAMADO,VENCIMIENTOS,MONTO_PAGADO,MONTO_DE_REASEGURO
0,2021,61,Invalidez total y permanente,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Flexible sin tasa garantizada,Femenino,1,1485500.71,0.0,1489939.28,0.0
1,2021,46,Asistencias,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-5800.39,0.0,0.00,0.0
2,2021,46,Fallecimiento,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-469891.60,0.0,0.00,0.0
3,2021,46,Gastos funerarios,Baja California,(OSTEO)ARTROSIS PRIMARIA GENERALIZADA,Vitalicio,Tradicional,Femenino,1,-46545.69,0.0,0.00,0.0
4,2021,64,Fallecimiento,Sonora,ABDOMEN AGUDO,Vitalicio,Flexible sin tasa garantizada,Femenino,1,495166.91,0.0,496646.44,0.0


=== COMISIONES ===


,ANIO,EDAD,PLAN_DE_LA_POLIZA,MODALIDAD_DE_LA_POLIZA,MONEDA,ENTIDAD,SEXO,FORMA_DE_VENTA,TIPO_DIVIDENDO,NUMERO_DE_ASEGURADOS,PRIMA_CEDIDA,COMISIONES_DIRECTAS,FONDO_DE_INVERSIÓN,FONDO_DE_ADMINISTRACION,MONTO_DE_DIVIDENDOS,MONTO_DE_RESCATE
0,2021,0,Temporal,Tradicional,Nacional,Morelos,Masculino,Agentes Persona Física,Sin dividendo,1,0.00,275.00,0,0,0.0,0
1,2021,0,Temporal,Tradicional,Nacional,Veracruz,Femenino,Agentes Persona Física,Sin dividendo,1,0.00,192.00,0,0,0.0,0
2,2021,0,Temporal,Tradicional,Nacional,Veracruz,Masculino,Agentes Persona Física,Sin dividendo,1,0.00,192.00,0,0,0.0,0
3,2021,0,Temporal,Deudores,Nacional,Chihuahua,Femenino,Agentes Persona Física,Sin dividendo,2,105.04,51.03,0,0,0.0,0
4,2021,0,Temporal,Deudores,Nacional,Chihuahua,Femenino,Agentes Persona Moral,Sin dividendo,3,269.63,130.99,0,0,0.0,0
